# HEPData stuff

Pre, post-fit

In [ ]:
import json
import numpy as np
import uproot
import plot_utils as utils
    
def read_post_fit_file(postFitFile, bins, regions):
   
    payload, payload_err = {}, {}
    for region in regions:
        
        payload[region], payload_err[region] = {}, {}

        for fit in ['fit_s', 'fit_b', 'prefit']:
                
            payload[region][fit], payload_err[region][fit] = {}, {}
            payload[region]['data'], payload_err[region]['data'] = {}, {}

            for bin in bins:

                name = 'shapes_{}/{};1'.format(fit, region+"_"+bin)
                
                payload[region][fit][bin] = postFitFile[name]['total_background;1'].values()[0]
                payload_err[region][fit][bin] = np.sqrt(postFitFile[name]['total_covar;1'].to_numpy()[0][0][0])

                payload[region]['data'][bin] = postFitFile[name]['data;1'].values()[1][0]
                payload_err[region]['data'][bin] = np.array(utils.compute_poisson_interval(payload[region]['data'][bin]))

    return payload, payload_err


def make_hepdata(_rpayload, _rpayload_err):

    bins = [
        'A', 'B', 'C', 'D', 'E', 'F_Fbin0','F_Fbin1','F_Fbin2','F_Fbin3','F_Fbin4', 'G', 'H', 'SR_SRbin0','SR_SRbin1','SR_SRbin2','SR_SRbin3','SR_SRbin4'
    ]
    bin_edges = [
        [10, 20], [20, 30], [30, float('inf')], [10, 20], [20, 30], [30, 40], [40, 50], [50, 60], [60, 80], [80, float('inf')], [10, 20], [20, 30], [30, 40], [40, 50], [50, 60], [60, 80], [80, float('inf')]
    ]
    out = {}
    out['bins'] = bins
    out['bin_edges'] = bin_edges

    # post fit values for each region
    values = np.array([_rpayload['SR']['fit_b'][bin] for bin in bins])
    errors = np.array([_rpayload_err['SR']['fit_b'][bin] for bin in bins])
    out['fit_b'] = values.tolist()
    out['fit_b_err'] = errors.tolist()

    # observed data for each region
    values = np.array([_rpayload['SR']['data'][b] for b in bins])
    errors = np.array([_rpayload_err['SR']['data'][bin] for bin in bins])
    errors[np.isnan(errors)] = 0
    out['data'] = values.tolist()
    out['data_err'] = np.array([values-errors[:,0], errors[:,1] - values]).T.tolist()
    
    return out

bins = [
    'SR_SRbin0','SR_SRbin1','SR_SRbin2','SR_SRbin3','SR_SRbin4',
    'F_Fbin0','F_Fbin1','F_Fbin2','F_Fbin3','F_Fbin4',
    'A', 'B', 'C', 'D', 'E', 'G', 'H',
]
regions = ['SR']

filename = "../cards/WH_2_1/cards-SUEP_mS125.000_mPhi3.000_T3.000_modegeneric/fitDiagnosticsTest.root"  # replace with your actual ROOT file path
file = uproot.open(filename)
payload, payload_err = read_post_fit_file(file, bins, regions)
hepdata = make_hepdata(payload, payload_err)

with open('Entry/hepdata_fit_wh.json', 'w') as f:
    json.dump(hepdata, f)

Signal cutflow table

In [ ]:
import numpy as np
import sys
sys.path.append("/home/submit/lavezzo/SUEP/SUEPCoffea_dask") # a bit hardcoded...
sys.path.append("/home/submit/lavezzo/SUEP/SUEPCoffea_dask/plotting")

from plotting.utils.loader import loader, getHistList
from plotting.utils import plot_utils as other_plot_utils

In [ ]:
local_hist_dir = "/ceph/submit/data/user/l/lavezzo/SUEP/outputs/"

files_signals = []
files_signals += getHistList(local_hist_dir, "WH_1_25_signal_2018_limits",  "/home/submit/lavezzo/SUEP/SUEPCoffea_dask/filelist/WH/list_2018_signal.txt", file_ext=".root")
cutflow_signals = loader(files_signals, by_year=False, only_cutflows=True)

Finished loading all files


In [ ]:
selections = [
    'cutflow_allTriggers_nominal',
    'cutflow_qualityFilters_nominal',
    'cutflow_orthogonality_nominal',
    'cutflow_oneTightLepton_nominal',
    'cutflow_oneAK4jet_nominal',
    'cutflow_MET20_nominal',
    'cutflow_twoTracksInCluster_nominal',
    'cutflow_PuppiMET_pt_>_30.0_SR',
    'cutflow_W_pt_>_60.0_SR',
    'cutflow_W_mt_>_30.0_SR',
    'cutflow_bjetSel_==_1.0_SR',
    'cutflow_deltaPhi_SUEP_W_>_1.5_SR',
    'cutflow_deltaPhi_SUEP_MET_>_1.5_SR',
    'cutflow_deltaPhi_lepton_SUEP_>_1.5_SR',
    'cutflow_ak4jets_inSUEPcluster_n_HighestPT_>=_1.0_SR',
    'cutflow_W_SUEP_BV_<_2.0_SR',
    'cutflow_deltaPhi_minDeltaPhiMETJet_MET_>_1.5_SR'
]
selection_labels = [
    "Triggers",
    "Quality filters",
    "Orthogonality",
    "One tight lepton",
    "One AK4 jet",
    "MET > 20",
    "One cluster",
    "$p^{miss}_T$ > 30 GeV & Reweighting",
    "$p^{W}_T$ > 60 GeV",
    "30 GeV < $m^{W}_T$ < 130 GeV",
    "b-jet selection",
    "$\Delta\phi(W, SUEP)$ > 1.5",
    "$\Delta\phi(MET, SUEP)$ > 1.5",
    "$\Delta\phi(lepton, SUEP)$ > 1.5",
    ">=1 AK4 jet in SUEP cluster",
    "$p^{W}_{T}/p^{SUEP}_T$ < 3",
    "min($\Delta\phi(MET, Jet)$) > 1.5"
]

In [ ]:
table = other_plot_utils.make_cutflow_table(
    cutflow_signals,
    cutflow_signals.keys(),
    selections,
    relative_efficiencies=True
    #selection_labels, 5
)
cumulative_effs = np.prod(table, axis=1)
table = np.hstack((table, np.array([cumulative_effs]).T))
selection_labels.append('Cumulative Efficiency')
table = np.round(table, 3)
table

array([[1.   , 0.363, 1.   , ..., 0.955, 0.783, 0.014],
       [1.   , 0.36 , 1.   , ..., 0.989, 0.835, 0.033],
       [1.   , 0.356, 1.   , ..., 0.997, 0.841, 0.065],
       ...,
       [1.   , 0.352, 1.   , ..., 0.998, 0.781, 0.075],
       [1.   , 0.377, 1.   , ..., 0.998, 0.786, 0.068],
       [1.   , 0.412, 0.999, ..., 0.999, 0.783, 0.052]])

In [ ]:
def get_signal_params(name):
    import re

    pattern = r'SUEP-WH-mS(\d+)_T(\d+\.\d+)_mPhi(\d+\.\d+)_(\w+)'

    # Use re.search to find the first occurrence of the pattern in the sample name
    match = re.search(pattern, name)

    # Extract the matched groups and convert them to the appropriate data types
    temp = float(match.group(3))
    mS = float(match.group(1))
    mPhi = float(match.group(2))
    decay = match.group(4)
    mAprime = plot_utils.mA[decay]

    # Return the extracted parameters as a tuple
    return mS, mPhi, temp, mAprime

signals = [get_signal_params(s) for s in cutflow_signals.keys()]

In [ ]:
import csv

hep_table = []
hep_table = [['$m_S$ [GeV]', '$m_{\phi}$ [GeV]', '$T_D$ [GeV]', '$m_{A\'}$ [GeV]'] + selection_labels]
for signal, cutflow in zip(signals, table):
    row = list(signal) + list(cutflow)
    hep_table.append(row)

with open('Entry/hepdata_cutflow_wh.csv', 'w', newline='') as f:
    writer = csv.writer(f)
    writer.writerows(hep_table)